[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/dynamo/dynamo.ipynb)

# A Dynamo in a Planet's Core

This example simulates a hydromagnetic dynamo in a planetary core, where stirred liquid metal amplifies a weak seed magnetic field until magnetic forces push back on the flow that sustains it. Because a conducting fluid drags magnetic field lines with it, a helical flow that stretches and twists those lines strengthens the field faster than ohmic resistance lets it decay. Once the induced currents grow strong enough, the magnetic force on them brakes the fluid shell between the solid inner core and the poorly conducting mantle, and the flow and field settle into trading energy back and forth.

In spacetime, spatial displacements, fluid velocities, and conduction 4-currents are intrinsic spatial directions, while a magnetic field is an extrinsic spatial plane reconstructed in each tetrahedron as the curl of edge potentials via Whitney forms. The term ***simplex gradient*** here means the spatial plane in a tetrahedron along which a corner's linear weight rises at unit rate from the opposite face, computed as `(face_patches & time) / (3 * prisms)`. Moving fluid meets the magnetic plane in the spatial electric form `velocities & Magnetic`, swept through time into a motional electric field that drives rest-frame conduction 4-currents via `-(Field & time).dual()`. Pairing the magnetic plane back with those 4-currents as `middle_field & currents` yields the Lorentz braking form on fluid velocities, whose alternating regressive pairing `(middle_field & currents) & velocity == -(velocity & middle_field) & currents` makes the mechanical power lost by the flow equal the electrical power gained by the field.

In Gibbs–Heaviside vector magnetohydrodynamics, the magnetic field and vorticity are represented as axial vectors $\mathbf{B} = \nabla \times \mathbf{A}$ and $\boldsymbol{\omega} = \nabla \times \mathbf{u}$, Ohm's law combines inductive and motional terms through a cross product $\mathbf{J} = \sigma(-\partial_t \mathbf{A} - \nabla \phi + \mathbf{u} \times \mathbf{B})$, and momentum balance couples self-advection and Lorentz braking through two further cross products, $\partial_t \mathbf{u} + \boldsymbol{\omega} \times \mathbf{u} + \nabla p = \nu \nabla^2 \mathbf{u} + \mathbf{J} \times \mathbf{B} + \mathbf{f}$, whose power cancellation relies on the vector triple-product identity $(\mathbf{J} \times \mathbf{B}) \cdot \mathbf{u} = -(\mathbf{u} \times \mathbf{B}) \cdot \mathbf{J}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import islice, tee

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import Sparse, SparseExtensor, spdiag
from numga.topology import Topology
from examples.electromagnetism.dynamo import render, scenarios
from examples.util.animation import save_figure, show_animation

mv = NumpyContext(ga).multivector
# Spatial planes within spacetime span x, y, and z.
space = Algebra("x+y+z+")
Scalar = ga.gatype.scalar()                         # Scalar+
Plane = ga.gatype.vector().derive.within(space)     # Vector+
Point = Plane.derive.dual                           # Antivector-
Field = ga.gatype.bivector()                        # Bivector+
Magnetic = Field.derive.within(space)               # Bivector+
Rate = Field.derive.within(space)                   # Bivector+
# Rest-frame observer 4-velocity and rest-frame conduction 4-current readout.
time = -mv.t.dual()                                 # Antivector-
electric_current = -(Field & time).dual()           # Point- <- Field+

## Tetrahedral Whitney operators in spacetime

Subdivided icosahedral shells coned to the origin partition the core and surrounding mantle into tetrahedra whose edges are spatial displacements. Sweeping a tetrahedron's opposite face patch through unit time and dividing by three spacetime prism volumes gives each corner's simplex gradient. At each centroid, an edge's Whitney 1-form blends its endpoint simplex gradients while twice their wedge gives its constant magnetic curl, so any edge potential reconstructs a source-free magnetic field across the ball. Pulling the vacuum excitation `volumes * Magnetic.dual()` back across the curl gives the magnetic stiffness on edge potentials.

In [ ]:
n_levels = scenarios.n_levels
n_shells = scenarios.n_shells
mantle_radius = scenarios.mantle_radius

In [ ]:
def whitney_operators(topology: Topology, vertices: Point) -> tuple[Scalar, SparseExtensor, SparseExtensor]:
    """Tetrahedral volumes (`Scalar+[T]`), centroid Whitney 1-forms (`(Plane+ <- Scalar+)[T <- E]`), and curls (`(Magnetic+ <- Scalar+)[T <- E]`) map edge circulations onto each tetrahedron."""
    edges = topology.d[0] * vertices                                                    # Point-[E]
    simplex_edges = topology.oriented[3, 1] * edges                                     # [6] Point-[T]
    prisms = ((simplex_edges[0] & simplex_edges[1] & simplex_edges[2]) / 6) & time      # Scalar-[T]
    sides = topology.boundary[2, 1] * edges                                             # [3] Point-[F]
    face_patches = topology.boundary[3, 2] * ((sides[0] & sides[1]) / 2)                # [4] Antibivector-[T]
    # Each corner's simplex gradient is the spacetime volume its opposite face sweeps over three prism volumes.
    gradients = (face_patches & time) / (3 * prisms)                                    # [4] Plane+[T]
    tails, heads = topology.corners[3, 1]                                               # [6], [6]
    # At each centroid an edge's Whitney form is a quarter of its endpoint gradient difference, and its curl is twice their wedge.
    centroid_forms = (gradients[heads] - gradients[tails]) / 4                          # [6] Plane+[T]
    whitney = (spdiag(centroid_forms * Scalar) * topology.oriented[3, 1]).sum(axis=0)   # (Plane+ <- Scalar+)[T <- E]
    curls = 2 * (gradients[tails] ^ gradients[heads])                                   # [6] Magnetic+[T]
    curl = (spdiag(curls * Scalar) * topology.oriented[3, 1]).sum(axis=0)               # (Magnetic+ <- Scalar+)[T <- E]
    return prisms.norm(), whitney, curl


def magnetic_stiffness(volumes: Scalar, curl: SparseExtensor) -> SparseExtensor:
    """Pulling the vacuum excitation back across the curl maps edge potentials to edge flux forms (`(Antiscalar- <- Scalar+)[E <- E]`)."""
    excitation = volumes * Magnetic.dual()                                              # (Antibivector- <- Magnetic+)[T]
    return curl.adjugate()(excitation(curl))                                            # (Antiscalar- <- Scalar+)[E <- E]


topology, vertices = scenarios.ball_complex(n_levels, n_shells)
centres = topology.incidence[3, 0].mean(axis=0) * vertices                              # Point-[T]
volumes, whitney, curl = whitney_operators(topology, vertices)
stiffness = magnetic_stiffness(volumes, curl)                                           # (Antiscalar- <- Scalar+)[E <- E]

## Incompressible flow modes

Fluid motion is confined to the spherical shell between the solid inner core and the mantle. Edge circulations on that shell map through Whitney 1-forms to fluid momentum, velocity, and vorticity in each tetrahedron, and subtracting the pressure gradient that balances vertex inflow projects any seed circulation onto incompressible flows. Laying the shell's slowest scalar modes along the three coordinate axes and solving the generalized eigenproblem of enstrophy against kinetic energy produces an orthonormal modal basis ordered by viscous decay rate.

In [ ]:
n_scalar_modes = scenarios.n_scalar_modes
n_modes = scenarios.n_modes
shown_modes = scenarios.shown_modes

In [ ]:
def flow_modes(fluid_topology: Topology, fluid_vertices: Point, embedding: Sparse,
               n_scalar_modes: int, n_modes: int) -> tuple[Scalar, Point, Rate]:
    """Incompressible eigenmodes of the fluid shell give viscous decay rates (`[n_modes] Scalar+`), velocities (`[n_modes] Point-[T]`), and vorticities (`[n_modes] Rate+[T]`)."""
    V_fluid, E_fluid, F_fluid, T_fluid = fluid_topology.n_elements
    volumes, whitney, curl = whitney_operators(fluid_topology, fluid_vertices)          # Scalar+[T_fluid], (Plane+ <- Scalar+)[T_fluid <- E_fluid], (Magnetic+ <- Scalar+)[T_fluid <- E_fluid]
    velocity_whitney = Plane.dual()(whitney)                                            # (Point- <- Scalar+)[T_fluid <- E_fluid]
    momentum = volumes * Point.dual()                                                   # (Plane+ <- Point-)[T_fluid]
    kinetic = velocity_whitney.adjugate()(momentum(velocity_whitney))                   # (Antiscalar- <- Scalar+)[E_fluid <- E_fluid]
    # Divergence at each vertex and the pressure stiffness that removes inflow, pinned at vertex zero.
    inflow = fluid_topology.d[0].adjugate() * kinetic                                   # (Antiscalar- <- Scalar+)[V_fluid <- E_fluid]
    pressure = inflow * fluid_topology.d[0]                                             # (Antiscalar- <- Scalar+)[V_fluid <- V_fluid]
    reference = mv.scalar((np.arange(V_fluid) == 0)[:, None]).field()                   # Scalar+[V_fluid]
    pinned_pressure = pressure + pressure.diagonal() * reference                        # (Antiscalar- <- Scalar+)[V_fluid <- V_fluid]
    vertex_volumes = spdiag((fluid_topology.incidence[3, 0].mean(axis=0).adjugate() * volumes) * Scalar.dual())  # (Antiscalar- <- Scalar+)[V_fluid <- V_fluid]
    _, scalar_modes = pressure.eigh(vertex_volumes, n_scalar_modes + 1)                 # [n_scalar_modes + 1] Scalar+[V_fluid]
    # Non-constant scalar modes at edge midpoints laid along x, y, and z and projected free of inflow.
    edge_means = fluid_topology.incidence[1, 0].mean(axis=0) * scalar_modes[1:]         # [n_scalar_modes] Scalar+[E_fluid]
    edge_spans = stack((mv.x, mv.y, mv.z))[:, None] & (fluid_topology.d[0] * fluid_vertices)  # [3, 1] Scalar+[E_fluid]
    seed_circulations = (edge_means * edge_spans).reshape(-1)                           # [3 * n_scalar_modes] Scalar+[E_fluid]
    solenoidal_flows = seed_circulations - fluid_topology.d[0] * pinned_pressure.solve(inflow(seed_circulations))  # [3 * n_scalar_modes] Scalar+[E_fluid]
    # Generalized eigenmodes of enstrophy against kinetic energy on the solenoidal subspace.
    basis_map = (solenoidal_flows * Scalar).batch().field(0, 1).adjoint()               # (Scalar+ <- Scalar+)[E_fluid <- 3 * n_scalar_modes]
    stiffness = magnetic_stiffness(volumes, curl)                                       # (Antiscalar- <- Scalar+)[E_fluid <- E_fluid]
    energy_form = (Scalar & (basis_map.adjugate()(kinetic(solenoidal_flows)).batch() * Scalar)).field(1, 2)     # (Scalar+ <- (Scalar+, Scalar+))[3 * n_scalar_modes, 3 * n_scalar_modes]
    enstrophy_form = (Scalar & (basis_map.adjugate()(stiffness(solenoidal_flows)).batch() * Scalar)).field(1, 2)  # (Scalar+ <- (Scalar+, Scalar+))[3 * n_scalar_modes, 3 * n_scalar_modes]
    decay_rates, mixtures = enstrophy_form.eigh(energy_form)                            # [3 * n_scalar_modes] Scalar+, [3 * n_scalar_modes] Scalar+[3 * n_scalar_modes]
    circulations = basis_map(mixtures[:n_modes])                                        # [n_modes] Scalar+[E_fluid]
    return decay_rates[:n_modes], (embedding * velocity_whitney)(circulations), (embedding * curl)(circulations)


fluid_topology, fluid_vertices, embedding = scenarios.outer_core(topology, vertices)
decay_rates, mode_velocities, mode_vorticities = flow_modes(
    fluid_topology, fluid_vertices, embedding, n_scalar_modes, n_modes
)
save_figure(render.modes_figure(centres, mantle_radius, mode_velocities[shown_modes]), "dynamo_modes");

Each cutaway panel shades the vertical cross-section from dark indigo at rest to amber at peak speed and traces cyan streamlines through the removed front hemisphere. From left to right, the slowest mode rotates the fluid shell nearly rigidly around the stationary inner core, while higher modes wind into finer poloidal and toroidal vortices.

## Induction, advection, and Lorentz braking

Pulling Ohm's conduction 4-current back across the Whitney 1-forms gives an edge conductance opposing potential changes and a motional drive from fluid crossing magnetic planes. Self-advection drives fluid momentum along `-(velocity & vorticity)`, whose pairing with `velocity` vanishes identically in every tetrahedron so advection redistributes kinetic energy among modes without net work. Keeping magnetic diffusion and the target helical flow's motional drive inside a single factorized Crank–Nicolson operator while stepping flow departures explicitly lets each step evaluate the midpoint Ampère 4-current and apply its Lorentz braking `middle_field & currents` back onto the modes.

In [ ]:
def induction_operators(volumes: Scalar, whitney: SparseExtensor, curl: SparseExtensor,
                        velocities: Point, conductivity: Scalar) -> tuple[SparseExtensor, SparseExtensor]:
    """Edge conductance and motional drive (`(Antiscalar- <- Scalar+)[E <- E]`) pull Ohm's conduction 4-current back onto the edges."""
    conduction = (volumes * conductivity) * electric_current                            # (Point- <- Field+)[T]
    electric = (Plane ^ mv.t)(whitney)                                                  # (Field+ <- Scalar+)[T <- E]
    conductance = whitney.adjugate()(conduction(electric))                              # (Antiscalar- <- Scalar+)[E <- E]
    # Moving fluid reads the magnetic plane as an electric field by meeting with velocity and sweeping through time.
    motional = ((velocities & Magnetic) ^ mv.t)(curl)                                   # (Field+ <- Scalar+)[T <- E]
    drive = whitney.adjugate()(conduction(motional))                                    # (Antiscalar- <- Scalar+)[E <- E]
    return conductance, drive


def flow_advection(volumes: Scalar, curl: SparseExtensor, mode_velocities: Point, mode_vorticities: Rate,
                   target_velocity: Point, amplitudes: Scalar, potential: Scalar) -> tuple[Field, Scalar]:
    """Departure motional field (`Field+[T]`) and self-advection mode rates (`[n_modes] Scalar+`) come from the current flow state."""
    velocity = (amplitudes * mode_velocities).sum(axis=0)                               # Point-[T]
    vorticity = (amplitudes * mode_vorticities).sum(axis=0)                             # Rate+[T]
    departure_field = ((velocity - target_velocity) & curl(potential)) ^ mv.t           # Field+[T]
    advection_rates = (-(volumes * (velocity & vorticity)) & mode_velocities).sites.sum()  # [n_modes] Scalar+
    return departure_field, advection_rates

In [ ]:
def dynamo_states(volumes: Scalar, whitney: SparseExtensor, curl: SparseExtensor,
                  decay_rates: Scalar, mode_velocities: Point, mode_vorticities: Rate,
                  conductivity: Scalar, viscosity: float, pull: float, target_amplitudes: Scalar,
                  potential: Scalar, dt: float, n_steps: int) -> Iterator[tuple[Scalar, Scalar]]:
    """Yields flow mode amplitudes (`[n_modes] Scalar+`) and edge potentials (`Scalar+[E]`) across coupled Crank–Nicolson and Adams–Bashforth steps."""
    conduction = (volumes * conductivity) * electric_current                            # (Point- <- Field+)[T]
    electric = (Plane ^ mv.t)(whitney)                                                  # (Field+ <- Scalar+)[T <- E]
    target_velocity = (target_amplitudes * mode_velocities).sum(axis=0)                 # Point-[T]
    conductance, drive = induction_operators(volumes, whitney, curl, target_velocity, conductivity)
    growth = drive - magnetic_stiffness(volumes, curl)                                  # (Antiscalar- <- Scalar+)[E <- E]
    implicit_operator = conductance - (dt / 2) * growth                                 # (Antiscalar- <- Scalar+)[E <- E]
    explicit_operator = conductance + (dt / 2) * growth                                 # (Antiscalar- <- Scalar+)[E <- E]
    damping = viscosity * decay_rates + pull                                            # [n_modes] Scalar+
    stirring_rates = pull * target_amplitudes                                           # [n_modes] Scalar+
    amplitudes = target_amplitudes                                                      # [n_modes] Scalar+
    departure_field, advection_rates = flow_advection(
        volumes, curl, mode_velocities, mode_vorticities, target_velocity, amplitudes, potential
    )
    previous_departure, previous_rates = departure_field, advection_rates               # Field+[T], [n_modes] Scalar+
    yield amplitudes, potential
    for _ in range(n_steps):
        extrapolated_field = 1.5 * departure_field - 0.5 * previous_departure           # Field+[T]
        next_potential = implicit_operator.solve(
            explicit_operator(potential) + dt * whitney.adjugate()(conduction(extrapolated_field))
        )                                                                               # Scalar+[E]
        potential_rate = (next_potential - potential) / dt                              # Scalar+[E]
        middle_field = curl((next_potential + potential) / 2)                           # Magnetic+[T]
        # Midpoint conduction 4-current balances Ampère's law between motional and inductive electric fields.
        currents = conduction(((target_velocity & middle_field) ^ mv.t) + extrapolated_field - electric(potential_rate))  # Point-[T]
        # Pairing the magnetic plane with the 4-current yields the Lorentz force 1-form on fluid velocities.
        braking_rates = ((middle_field & currents) & mode_velocities).sites.sum()       # [n_modes] Scalar+
        numerator_amplitudes = amplitudes + dt * (
            1.5 * advection_rates - 0.5 * previous_rates + braking_rates + stirring_rates - damping * amplitudes / 2
        )                                                                               # [n_modes] Scalar+
        amplitudes = numerator_amplitudes / (1 + (dt / 2) * damping)                    # [n_modes] Scalar+
        potential = next_potential                                                      # Scalar+[E]
        previous_departure, previous_rates = departure_field, advection_rates
        departure_field, advection_rates = flow_advection(
            volumes, curl, mode_velocities, mode_vorticities, target_velocity, amplitudes, potential
        )
        yield amplitudes, potential

## Saturated dynamo and cutaway flow

With high conductivity in the outer core and hundredfold lower conductivity in the mantle so magnetic flux can escape toward the surface, helical stirring of two orthogonal Chandrasekhar–Kendall spheromaks amplifies a weak uniform seed field until Lorentz braking saturates the dynamo. Stepping the full run and streaming the saturated top-view animation takes about a minute.

In [ ]:
reynolds = scenarios.reynolds
mantle_conductance = scenarios.mantle_conductance
viscosity = scenarios.viscosity
pull = scenarios.pull
seed_field = scenarios.seed_field                                               # Magnetic+
dt = scenarios.dt
n_steps = scenarios.n_steps
n_skipped_steps = scenarios.n_skipped_steps
stride = scenarios.stride
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
radii = centres.antinorm()                                                      # Scalar+[T]
conductivity = mv.scalar(np.where(radii < 1, reynolds, reynolds * mantle_conductance)[:, None]).field()  # Scalar+[T]
target_momentum = volumes * scenarios.helical_flow(topology, vertices).dual()   # Plane+[T]
target_amplitudes = (target_momentum & mode_velocities).sites.sum()             # [n_modes] Scalar+
initial_potential = scenarios.seed_potential(topology, vertices, seed_field)    # Scalar+[E]
top_stream, cutaway_stream, energy_stream = tee(
    dynamo_states(
        volumes, whitney, curl, decay_rates, mode_velocities, mode_vorticities,
        conductivity, viscosity, pull, target_amplitudes, initial_potential, dt, n_steps,
    ),
    3,
)


def saturated_frames(states: Iterable[tuple[Scalar, Scalar]]) -> Iterator[tuple[Magnetic, Point]]:
    """Yields tetrahedral magnetic planes (`Magnetic+[T]`) and velocities (`Point-[T]`) across the saturated window."""
    for amplitudes, potential in islice(states, n_skipped_steps, None, stride):
        yield curl(potential), (amplitudes * mode_velocities).sum(axis=0)


n_frames = (n_steps - n_skipped_steps) // stride + 1
show_animation(render.lines_animation(centres, saturated_frames(top_stream)), "dynamo_lines", frame_duration_ms, n_frames=n_frames);

Seen from above the pole inside a faint translucent outer-core sphere, cyan streamlines with downstream chevrons wind through white magnetic field lines that twist and reconnect as the saturated flow carries them. Slicing the same saturated states along a vertical plane through the planet's centre takes about half a minute to render.

In [ ]:
show_animation(render.cutaway_animation(centres, mantle_radius, saturated_frames(cutaway_stream)), "dynamo", frame_duration_ms, n_frames=n_frames);

Cyan surface streaks on the cut plane fade downstream across the fluid shell while remaining absent in the solid inner core and surrounding mantle, and white magnetic field tubes arch out of the core through the poorly conducting mantle into the removed front hemisphere.

## Energy exchange between flow and field

Orthonormality of the velocity modes makes kinetic energy half the squared norm of the mode amplitudes, while pairing edge potentials with their magnetic stiffness gives twice the magnetic energy stored across the core and mantle.

In [ ]:
n_energy_frames = n_steps // stride + 1
amplitudes, potentials = map(stack, zip(*islice(energy_stream, 0, None, stride)))  # [n_energy_frames, n_modes] Scalar+, [n_energy_frames] Scalar+[E]
kinetic_energy = (amplitudes * amplitudes).sum(axis=-1) / 2                     # [n_energy_frames] Scalar+
magnetic_energy = (potentials & stiffness(potentials)).sites.sum() / 2          # [n_energy_frames] Scalar+
times = np.arange(0, n_steps + 1, stride) * dt                                  # [n_energy_frames]
save_figure(render.energies_figure(times, kinetic_energy, magnetic_energy), "dynamo_energies");

The cyan curve shows the fluid kinetic energy and the orange curve shows the magnetic energy on a logarithmic scale. During the kinematic stage, magnetic energy grows exponentially across several orders of magnitude while the stirred flow remains undisturbed; once Lorentz braking becomes comparable to the stirring, kinetic energy drops and the two curves trade energy around saturation.

In [ ]:
# checks
# Modal kinetic energy matches volume-integrated momentum paired with velocity.
final_velocity = (amplitudes[-1] * mode_velocities).sum(axis=0)                 # Point-[T]
volume_energy = (volumes * (final_velocity.dual() & final_velocity)).sites.sum() / 2  # Scalar+
assert (volume_energy - kinetic_energy[-1]).abs() < 1e-10 * kinetic_energy[-1]
# Self-advection transfers energy among modes without net work.
target_velocity = (target_amplitudes * mode_velocities).sum(axis=0)             # Point-[T]
_, advection_rates = flow_advection(
    volumes, curl, mode_velocities, mode_vorticities, target_velocity, amplitudes[-1], potentials[-1]
)
power_scale = ((advection_rates * advection_rates).sum() * (amplitudes[-1] * amplitudes[-1]).sum()).square_root()
assert (amplitudes[-1] * advection_rates).sum().abs() < 1e-10 * power_scale
# Magnetic flux through every closed tetrahedral boundary vanishes (d2 * d1 == 0).
face_fluxes = topology.d[1] * potentials[-1]                                    # Scalar+[F]
assert ((topology.d[2] * face_fluxes).norm().batch() < 1e-10 * face_fluxes.norm().batch().max()).all()
# The seed magnetic energy amplifies by orders of magnitude and brakes the flow.
assert magnetic_energy[-1] > 100 * magnetic_energy[0]
assert kinetic_energy.min() < 0.75 * kinetic_energy[0]